<a href="https://colab.research.google.com/github/AyanamiiiiRei/AyanamiiiiRei.github.io/blob/main/lab_song_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [3]:
# Download data
!wget -O training_data.csv http://handsonml.control.lth.se/data/training_data.csv
!wget -O songs_to_classify.csv http://handsonml.control.lth.se/data/songs_to_classify.csv

--2026-09-28 16:36:16--  http://handsonml.control.lth.se/data/training_data.csv
Resolving handsonml.control.lth.se (handsonml.control.lth.se)... 130.235.83.36, 2001:6b0:16:248:2406:1311:4200:0
Connecting to handsonml.control.lth.se (handsonml.control.lth.se)|130.235.83.36|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 54950 (54K) [text/csv]
Saving to: ‘training_data.csv’

training_data.csv   100%[===================>]  53.66K   167KB/s    in 0.3s    

2026-09-28 16:36:17 (167 KB/s) - ‘training_data.csv’ saved [54950/54950]

--2026-09-28 16:36:17--  http://handsonml.control.lth.se/data/songs_to_classify.csv
Resolving handsonml.control.lth.se (handsonml.control.lth.se)... 130.235.83.36, 2001:6b0:16:248:2406:1311:4200:0
Connecting to handsonml.control.lth.se (handsonml.control.lth.se)|130.235.83.36|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14306 (14K) [text/csv]
Saving to: ‘songs_to_classify.csv’

songs_to_classify.c 100%[==========

In [4]:
# Load data
train = pd.read_csv("training_data.csv")
test = pd.read_csv("songs_to_classify.csv")
train.shape, test.shape

((750, 14), (200, 13))

In [5]:
# Inspect data
train.sample(5)

,acousticness,danceability,duration,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,label
675,0.82100,0.541,211426,0.193,0.000000,0,0.0999,-11.957,1,0.0433,117.470,3,0.533,1
220,0.00118,0.610,180000,0.969,0.074300,0,0.1840,-3.552,1,0.0556,127.976,4,0.236,0
644,0.60800,0.828,228693,0.523,0.000025,1,0.1080,-13.174,0,0.0417,121.580,4,0.778,1
127,0.09730,0.840,202200,0.707,0.000000,9,0.2700,-6.957,1,0.0514,129.532,4,0.697,0
746,0.00197,0.487,213000,0.867,0.006020,10,0.0968,-3.293,0,0.0543,160.048,4,0.403,0


In [9]:
# Feature Preprocessing Pipeline with RobustScaler and OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

# 1. 定义不同特征列的类别
numeric_features = [
    'acousticness', 'danceability', 'duration', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'speechiness',
    'tempo', 'valence'
]
categorical_features = ['key', 'time_signature']
binary_features = ['mode']

# 2. 构建预处理转换器 (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

# 3. 在训练集上拟合并转换数据 (Fit & Transform)
X_train_processed = preprocessor.fit_transform(train)
y_train = train['label'].values

# 4. 在测试集上仅做转换 (Transform only, 避免数据泄露)
X_test_processed = preprocessor.transform(test)

# 5. 查看预处理后的数据维度
print("=== 数据预处理完成 ===")
print(f"原始训练集特征维度: {train[numeric_features + categorical_features + binary_features].shape}")
print(f"处理后训练集特征维度: {X_train_processed.shape}")
print(f"处理后测试集特征维度: {X_test_processed.shape}")

=== 数据预处理完成 ===
原始训练集特征维度: (750, 13)
处理后训练集特征维度: (750, 27)
处理后测试集特征维度: (200, 27)


In [5]:
# select which features to use
features = ['danceability','key','loudness','instrumentalness','liveness']
X_train = train.loc[:,features].values
y_train = train.loc[:,'label'].values
X_test = test.loc[:,features].values

In [6]:
# Normalize data. Can also be done using sklearn methods such as
# MinMaxScaler() or StandardScaler()
X_trainn = X_train*1/np.max(np.abs(X_train), axis=0)
X_testn = X_test*1/np.max(np.abs(X_train), axis=0)

In [7]:
# note: all inputs/features are treated as quantitative/numeric
# some of the features are perhaps more sensible to treat as
# qualitative/cathegorical. For that sklearn preprocessing methods
# such as OneHotEncoder() can be used

# define the k-NN model. To set n_neighbors in a systematic way, use cross validation!
knnmodel = KNeighborsClassifier(n_neighbors=5)

# feed it with data and train it
knnmodel.fit(X_trainn, y_train)

# make predictions
predictions = knnmodel.predict(X=X_testn)
print(predictions)

[0 1 0 1 0 0 0 1 0 0 1 0 0 1 1 1 1 0 1 1 0 0 1 0 0 1 1 0 0 1 1 1 1 1 0 1 1
 1 0 1 0 1 0 1 1 1 0 1 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0 0 1 1 1 0 1 1 1 0 1 1
 1 1 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 0 0 0 1 0 1 1 1 1 0 0 1 0 0 1 0 0 0 1 1
 1 0 1 1 0 0 1 0 0 0 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 0 1 1 1 0 0
 1 0 1 1 1 0 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 1 1 1 1
 1 1 0 0 0 1 0 0 0 1 1 1 1 1 1]
